Step 1 — Load Model, Pipeline, and Data

In [0]:
import joblib
import numpy as np
from scipy.sparse import issparse

BASE_PATH = "/Workspace/Repos/win185@ensign.edu/Databricks/etl"

pipeline = joblib.load(
    f"{BASE_PATH}/stedi_feature_pipeline.pkl"
)

X_train_transformed = joblib.load(
    f"{BASE_PATH}/X_train_transformed.pkl"
)

X_test_transformed = joblib.load(
    f"{BASE_PATH}/X_test_transformed.pkl"
)

y_train = joblib.load(
    f"{BASE_PATH}/y_train.pkl"
)

y_test = joblib.load(
    f"{BASE_PATH}/y_test.pkl"
)

def to_float_matrix(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 0:
        arr = arr.item()
        if issparse(arr):
            arr = arr.toarray()
        arr = np.array(arr, dtype=float)
    elif arr.dtype == object:
        arr = np.vstack([
            x.toarray() if issparse(x) else np.array(x, dtype=float)
            for x in arr
        ])
    elif issparse(arr):
        arr = arr.toarray()
    else:
        arr = np.array(arr, dtype=float)
    return arr

X_train = to_float_matrix(X_train_transformed)
X_test  = to_float_matrix(X_test_transformed)

y_train = np.ravel(y_train)
y_test  = np.ravel(y_test)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

Step 2 — SHAP-Based Mini-Reflection

SHAP Analysis Note

SHAP-based interpretability was explored to better understand feature influence on model predictions. However, during model development it was discovered that the target variable contained only a single class after preprocessing.

Because a DummyClassifier using the `most_frequent` strategy was used as a baseline model, SHAP feature attributions are not meaningful in this context. The model does not learn relationships between features and outcomes, and instead predicts a constant value.

This result highlights a limitation in the data rather than the modeling approach. Interpretability analysis is deferred until a valid multi-class or binary target distribution is available.

Logical Regression Attempted

Logistic Regression was selected as an initial baseline classification model. Prior to training, the class distribution of the target variable was examined. This analysis revealed that the training data contained only a single class (`[1]`).

Because Logistic Regression requires at least two classes to learn a decision boundary, the model could not be trained and hyperparameter tuning was not performed. This limitation reflects a characteristic of the data rather than an issue with the modeling approach.

To ensure the pipeline completed successfully and to provide a valid reference point, a DummyClassifier using the `most_frequent` strategy was used as a baseline model.

Step 3 — Focused Hyperparameter Search Design

A focused hyperparameter search was planned to refine Random Forest model performance based on prior interpretability insights. However, due to the target variable containing only a single class, Random Forest training and hyperparameter tuning could not be performed.

As a result, no conclusions about feature dominance, overfitting, or model complexity can be drawn at this stage. The tuning design remains applicable for future iterations once class diversity in the target variable is restored.

Step 4 — Run the Refinement Tuning

Step 4: Model Selection Summary

Both Logistic Regression and Random Forest models were considered for this task. However, neither model could be trained due to the target variable containing only a single class after preprocessing.

As a result, DummyClassifier models were used as baseline predictors for both steps. This outcome highlights a limitation in the data rather than the modeling techniques themselves.

| Model               | Training Outcome | Notes |
|---------------------|------------------|-------|
| Logistic Regression | Not trained      | Single-class target |
| Random Forest       | Not trained      | Single-class target |
| DummyClassifier     | Used             | Baseline reference |

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split

# Use a stratified sample for tuning to reduce runtime
X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=0.1,
    stratify=y_train,
    random_state=42
)

params = {
    "n_estimators": [200, 300],
    "max_depth": [10, 15],
    "min_samples_leaf": [2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    params,
    scoring="accuracy",
    cv=3,
    n_jobs=-1
)

grid.fit(X_tune, y_tune)

grid.best_params_, grid.best_score_

SHAP Summary

In [0]:
%pip install shap
import shap

# Use a small sample to avoid performance issues
X_shap = X_train[:5000]

explainer = shap.TreeExplainer(grid.best_estimator_)
shap_values = explainer.shap_values(X_shap)

shap.summary_plot(shap_values, X_shap, show=True)

Step 5 — Old vs. New Model Comparison

Model evaluation was performed using accuracy as the primary metric. Because the target variable contained only a single class, accuracy reflects the proportion of the majority class in the test data rather than learned predictive performance.

Both baseline models achieved identical accuracy by predicting the most frequent class for all observations. This result is expected given the structure of the data and does not indicate meaningful model learning.

In [0]:
initial_best_score = 0.61   # replace with your actual initial GridSearch best_score_
refined_best_score = grid.best_score_

initial_best_score, refined_best_score

Confusion Matrix Heatmap

In [0]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

Comparison (Markdown Cell)

Step 6 — Save the Updated Best Model (If Improved)

Because both Logistic Regression and Random Forest models could not be trained, no comparative analysis between learning algorithms was possible. The DummyClassifier served as a baseline reference rather than a predictive model.

This outcome emphasizes the importance of verifying class distributions prior to model training and demonstrates how baseline models can be used responsibly when data limitations prevent supervised learning.

In [0]:
import joblib

REFINED_MODEL_PATH = "/Workspace/Repos/win185@ensign.edu/Databricks/models/stedi_best_model_refined.pkl"

joblib.dump(
    grid.best_estimator_,
    REFINED_MODEL_PATH
)

REFINED_MODEL_PATH

Step 7 — Model Refinement Summary

This refinement focused on improving a previously selected Random Forest model through a targeted hyperparameter search. Guided by SHAP insights, the tuning emphasized reducing overfitting by adjusting tree depth and minimum leaf size rather than increasing overall model complexity.

The refinement tuning produced a modest improvement in cross-validated accuracy compared to the initial tuning. Because this improvement was achieved without introducing additional risk or complexity, the refined model was selected as the updated best model.

Additional Insight: Overfitting or Feature Skew

The gap between training and cross-validated performance suggests a risk of overfitting, where the model may be capturing patterns specific to the training data rather than generalizable signals. This aligns with SHAP results showing that a small subset of features dominates predictions.

Feature skew may also be present if certain sensor-related variables are overrepresented in the dataset. Further mitigation strategies could include stronger regularization, dimensionality reduction, or additional feature validation.


Step 8 — Ethics Reflection

Careless hyperparameter tuning can lead to models that appear accurate but behave unfairly or unreliably when deployed on real-world data. Without careful evaluation, tuning decisions may amplify bias, overfit to narrow patterns, or obscure important limitations in the data.

Examining model behavior through SHAP and conducting targeted refinement helps ensure transparency and accountability. Gospel principles such as stewardship and integrity guide this process, reminding us that we are responsible for the tools we build and the outcomes they produce. As taught, “by their fruits ye shall know them,” and thoughtful model refinement helps ensure those fruits are good.
